Setup data paths + compute class counts


In [7]:
import os
import shutil
import pandas as pd
import numpy as np

# Paths
csv_path = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\Data_Entry_2017.csv"
current_root = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull"   # folder containing pneumonia/ and not_pneumonia/
output_root = "dataset"

# Load metadata
df = pd.read_csv(csv_path)

# Make binary label
df["label"] = df["Finding Labels"].apply(
    lambda x: "pneumonia" if "Pneumonia" in str(x) else "normal"
)
# Unique patients
patients = df["Patient ID"].drop_duplicates().tolist()

# Shuffle patients reproducibly
rng = np.random.default_rng(42)
rng.shuffle(patients)

# CheXNet-style patient counts
n = len(patients)

n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_patients = set(patients[:n_train])
val_patients = set(patients[n_train:n_train + n_val])
test_patients = set(patients[n_train + n_val:])

def assign_split(pid):
    if pid in train_patients:
        return "train"
    elif pid in val_patients:
        return "val"
    elif pid in test_patients:
        return "test"
    return None

df["split"] = df["Patient ID"].apply(assign_split)

# Copy files from current folders into final split folders
for _, row in df.iterrows():
    filename = row["Image Index"]
    label = row["label"]
    split = row["split"]

    if split is None:
        continue

    src = os.path.join(current_root, label, filename)
    dst_dir = os.path.join(output_root, split, label)
    dst = os.path.join(dst_dir, filename)

    os.makedirs(dst_dir, exist_ok=True)

    if os.path.exists(src):
        shutil.copy2(src, dst)
    else:
        print(f"Missing: {src}")



Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000013_010.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000032_012.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000056_000.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000061_012.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000061_015.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000144_001.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000150_002.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000165_001.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000193_019.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000211_013.png
Missing: C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\pneumonia\00000211_018.png
Missing: C

In [6]:
print(os.listdir(current_root))

['ARXIV_V5_CHESTXRAY.pdf', 'BBox_List_2017.csv', 'Data_Entry_2017.csv', 'FAQ_CHESTXRAY.pdf', 'LOG_CHESTXRAY.pdf', 'normal', 'pneumonia', 'README_CHESTXRAY.pdf', 'test_list.txt', 'train_val_list.txt']


In [ ]:
import random
from pathlib import Path

# 1. Setup paths
base_path = Path("C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull")
class_a_dir = base_path / 'NORMAL'
class_b_dir = base_path / 'PNEUMONIA'

# 2. Get lists of all images
exts = ['.jpg', '.jpeg', '.png']
normal_imgs = [f for f in class_a_dir.glob('*') if f.suffix.lower() in exts]
pneumonia_imgs = [f for f in class_b_dir.glob('*') if f.suffix.lower() in exts]

# 3. Determine the minimum count
min_samples = min(len(normal_imgs), len(pneumonia_imgs))

print(f"Normal images: {len(normal_imgs)}")
print(f"Pneumonia images: {len(pneumonia_imgs)}")
print(f"Balancing dataset to {min_samples} samples per class...")

# # 4. Undersample the larger class
# balanced_normal = random.sample(normal_imgs, min_samples)
# balanced_pneumonia = random.sample(pneumonia_imgs, min_samples)

# # 5. Combine into a final list for your model
# final_dataset = balanced_normal + balanced_pneumonia
# random.shuffle(final_dataset) # Mix them up!

# print(f"Balanced Dataset Size: {len(final_dataset)}")

Normal images: 7027
Pneumonia images: 7027
Balancing dataset to 7027 samples per class...


Copy balanced images into respective dataset folder

In [2]:
import shutil
from pathlib import Path

# Define your destination
target_base = Path('RSNA_balanced')

# Define a helper function to save the lists we created earlier
def save_balanced_class(image_list, class_name):
    # Create the folder: train_balanced/NORMAL (or PNEUMONIA)
    target_dir = target_base / class_name
    target_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Copying {len(image_list)} images to {target_dir}...")
    
    for img_path in image_list:
        # Copy file from original location to new location
        shutil.copy(img_path, target_dir / img_path.name)

# Run the saving process
save_balanced_class(balanced_normal, 'NORMAL')
save_balanced_class(balanced_pneumonia, 'PNEUMONIA')

print("Done! Your balanced dataset is ready.")

Copying 6012 images to RSNA_balanced\NORMAL...
Copying 6012 images to RSNA_balanced\PNEUMONIA...
Done! Your balanced dataset is ready.


Balance chest x rays, resize to 224x224, save JPEG


In [1]:
import random
from PIL import Image
from pathlib import Path

# --- 1. SET YOUR PATHS ---
# This is where your original folders (NORMAL, PNEUMONIA) live
input_base = Path('Kermany_Balanced') 
# This is where the processed images will go
target_base = Path('Kermany_Balancedv2')

# --- 2. THE COLLECTOR & BALANCER ---
def get_balanced_lists(source_path):
    # Support common image formats
    extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    
    # Grab all file paths for both categories
    normal_paths = [f for f in (source_path / 'NORMAL').glob('*') 
                    if f.suffix.lower() in extensions]
    pneumonia_paths = [f for f in (source_path / 'PNEUMONIA').glob('*') 
                       if f.suffix.lower() in extensions]
    
    # Find the smaller count to balance the dataset
    min_count = min(len(normal_paths), len(pneumonia_paths))
    print(f"Balancing to {min_count} images per class...")
    
    # Randomly select min_count images from each list
    balanced_n = random.sample(normal_paths, min_count)
    balanced_p = random.sample(pneumonia_paths, min_count)
    
    return balanced_n, balanced_p

# --- 3. THE PROCESSOR (Your Function) ---
def process_and_save(image_list, class_name):
    target_dir = target_base / class_name
    target_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Processing {len(image_list)} images for {class_name}...")
    
    for img_path in image_list:
        try:
            with Image.open(img_path) as img:
                img = img.convert('RGB')
                img = img.resize((224, 224), Image.LANCZOS)
                
                # Save as .jpg
                save_path = target_dir / f"{img_path.stem}.jpg"
                img.save(save_path, 'JPEG', quality=95)
        except Exception as e:
            # This captures the "No such file" or "Corrupt image" errors
            print(f"Error processing {img_path.name}: {e}")

# --- 4. EXECUTION ---
if input_base.exists():
    # Step A: Get the balanced lists of paths
    balanced_normal, balanced_pneumonia = get_balanced_lists(input_base)
    
    # Step B: Process them
    process_and_save(balanced_normal, 'NORMAL')
    process_and_save(balanced_pneumonia, 'PNEUMONIA')
    
    print("\Success! All images resized to 224x224 and saved.")
else:
    print(f"Error: The folder '{input_base}' was not found. check your path!")

<>:61: SyntaxWarning: invalid escape sequence '\S'
<>:61: SyntaxWarning: invalid escape sequence '\S'
C:\Windows\Temp\ipykernel_6552\2842496224.py:61: SyntaxWarning: invalid escape sequence '\S'
  print("\Success! All images resized to 224x224 and saved.")


Balancing to 1341 images per class...
Processing 1341 images for NORMAL...
Processing 1341 images for PNEUMONIA...
\Success! All images resized to 224x224 and saved.


Merge datasets into Combined_Dataset and create CSV index

In [ ]:
import shutil
import pandas as pd
from pathlib import Path

base_dir = Path('.')
target_base = base_dir / 'Combined_Dataset'

sources = [
    'Kermany_Balancedv2',
    'nih_sampled',
    'RSNA_balanced'
]

classes = ['NORMAL', 'PNEUMONIA']

records = []

# create dirs
for cls in classes:
    (target_base / cls).mkdir(parents=True, exist_ok=True)

# merge
for source in sources:
    for cls in classes:
        src_path = base_dir / source / cls
        
        if not src_path.exists():
            continue
        
        for img_path in src_path.glob('*'):
            if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                continue
            
            new_name = f"{source}_{img_path.name}"
            dst_path = target_base / cls / new_name
            
            shutil.copy2(img_path, dst_path)

            records.append({
                "image_path": str(dst_path),
                "label": cls,
                "source": source
            })

df = pd.DataFrame(records)
df.to_csv("combined_master.csv", index=False)

print("Merge complete")

Merge complete


Split combined data into train/val/test (80/10/10)

In [8]:
from sklearn.model_selection import train_test_split

# load
df = pd.read_csv("combined_master.csv")

# 80% train, 20% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

# split temp into val/test (10/10)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)

# save
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("80/10/10 split done")

80/10/10 split done


Organize split files into Master_Dataset folder structure

In [9]:
import os
import shutil

# The root folder where you want to build the new structure
base_dest_dir = "Master_Dataset"

def organize_images(df, split_name):
    print(f"Processing {split_name} split...")
    
    for index, row in df.iterrows():
        # 1. Get the source path from CSV and fix slashes for your OS
        # This changes 'Combined_Dataset\NORMAL\img.jpg' to work on any system
        src_relative_path = row['image_path'].replace('\\', os.sep)
        
        # 2. Get the label and filename
        label = row['label']
        filename = os.path.basename(src_relative_path)
        
        # 3. Define the destination: Master_Dataset/train/NORMAL/image.jpg
        target_dir = os.path.join(base_dest_dir, split_name, label)
        dst_path = os.path.join(target_dir, filename)
        
        # 4. Create the folder if it doesn't exist
        os.makedirs(target_dir, exist_ok=True)
        
        # 5. Copy the file
        if os.path.exists(src_relative_path):
            shutil.copy(src_relative_path, dst_path)
        else:
            # If the file isn't found, try looking for it in the current directory 
            # if Combined_Dataset is a folder right next to your script
            print(f"File not found: {src_relative_path}")

# Run the process for your three dataframes
organize_images(train_df, "train")
organize_images(val_df, "val")
organize_images(test_df, "test")

print("\nDone! Your images are now organized in 'Master_Dataset'")

Processing train split...
Processing val split...
Processing test split...

✅ Done! Your images are now organized in 'Master_Dataset'


Compute train set per-channel mean and std for normalization

E.g.
mean = [mean_R, mean_G, mean_B]
std  = [std_R, std_G, std_B]

In [13]:
from PIL import Image
import numpy as np
from tqdm import tqdm

train_df = pd.read_csv("train.csv")

mean = np.zeros(3)
std = np.zeros(3)
n_pixels = 0

for path in tqdm(train_df['image_path']):
    img = Image.open(path).convert('RGB')
    img = np.array(img) / 255.0
    
    n_pixels += img.shape[0] * img.shape[1]
    
    mean += img.sum(axis=(0, 1))
    std += (img ** 2).sum(axis=(0, 1))

mean /= n_pixels
std = np.sqrt(std / n_pixels - mean ** 2)

print("Mean:", mean)
print("Std:", std)

  0%|          | 0/14054 [00:00<?, ?it/s]

100%|██████████| 14054/14054 [01:03<00:00, 222.69it/s]

Mean: [0.51149503 0.51149503 0.51149503]
Std: [0.24570894 0.24570894 0.24570894]
